# Supervisor 경량화 독립 실험

symPO 의 supervisor(`task_match`) 단계를 오픈소스 LLM 으로 교체했을 때의
**정확도 손실 vs 속도 증가** trade-off 를 측정한다.

- **Baseline (gold)**: Gemini (frontier)
- **후보 OSS** (2026-04 최신 시리즈): Qwen3.5-8B-Instruct / Qwen3.5-4B-Instruct / Gemma-4-E4B-it / Gemma-4-E2B-it / Phi-4-mini-instruct (4-bit NF4)
- **입력**: `benchmark_cases.json` (업로드 시) 또는 인라인 fallback
- **지표**: JSON parse rate · called_agents Jaccard · allocations Jaccard · latency
- **판정**: `efficiency = speedup / quality_loss` — ≥2 합당 · 1~2 조건부 · <1 비합당

실행 전 Runtime → Change runtime type → **T4 GPU** 선택. 프로젝트 코어는 전혀 건드리지 않는 독립 실험이다.

In [ ]:
# 1) 의존성 설치 (Colab 세션 최초 1회)
!pip -q install --upgrade transformers==4.46.3 accelerate==1.1.1 bitsandbytes==0.44.1
!pip -q install google-generativeai pandas matplotlib

In [ ]:
# 2) imports + API key
import os, json, time, re, gc
from getpass import getpass
import pandas as pd
import matplotlib.pyplot as plt
import torch

if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass('GOOGLE_API_KEY: ')

# Gemma / Llama 등 gated 모델을 후보에 넣을 때만 필요. 비워도 됨.
if not os.environ.get('HF_TOKEN'):
    tok_in = getpass('HF_TOKEN (gated 모델용, 없으면 Enter): ')
    if tok_in.strip():
        os.environ['HF_TOKEN'] = tok_in.strip()

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if torch.cuda.is_available() else '-')

In [ ]:
# 3) benchmark cases 로드 — 업로드된 파일 우선, 없으면 인라인 fallback
CASES_PATH = 'benchmark_cases.json'
if os.path.exists(CASES_PATH):
    with open(CASES_PATH, 'r', encoding='utf-8') as f:
        CASES = json.load(f)
    print(f'Loaded {len(CASES)} cases from {CASES_PATH}')
else:
    # 최소 fallback 케이스 1개 — 실전 사용 시 benchmark_cases.json 업로드 권장
    CASES = [{
        'case_id': 'tm_fallback',
        'task': 'task_match',
        'prompt': (
            '[Task 매칭 Manager]\n팀원 2명에게 L3 태스크 2개를 배정하세요.\n'
            '- MBR-001: Python/Django\n- MBR-002: React\n\n'
            '- L3-01: API 개발 (Backend)\n- L3-02: UI 개발 (Frontend)\n\n'
            '```json\n{"called_agents": [...], "allocations": {"L3-01": ["MBR-..."]},'
            ' "calling_context": {...}}\n```'
        )
    }]
    print('benchmark_cases.json 없음 → 인라인 fallback 1개 사용')

for c in CASES:
    print(f"  - {c['case_id']}: {c.get('description', '(no description)')}")

In [ ]:
# 4) Gemini baseline — gold standard 생성
import google.generativeai as genai
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

# 프로젝트 기본: gemini-3.1-flash-lite-preview. Colab 공개 API 안정성 위해 기본은 2.5-flash.
GEMINI_MODEL = 'gemini-2.5-flash'
model_g = genai.GenerativeModel(GEMINI_MODEL)

def run_gemini(prompt: str, retries: int = 2):
    last_err = None
    for _ in range(retries + 1):
        try:
            t0 = time.time()
            resp = model_g.generate_content(prompt, generation_config={'temperature': 0.3, 'max_output_tokens': 2048})
            return resp.text, time.time() - t0
        except Exception as e:
            last_err = e; time.sleep(2)
    return f'[ERROR] {last_err}', 0.0

gemini_rows = []
for c in CASES:
    text, lat = run_gemini(c['prompt'])
    gemini_rows.append({'case_id': c['case_id'], 'model': GEMINI_MODEL, 'text': text, 'latency_sec': lat})
    print(f"  [{c['case_id']}] gemini: {lat:.2f}s, {len(text)} chars")

ALL_RESULTS = {GEMINI_MODEL: gemini_rows}

In [ ]:
# 5) OSS 모델 로더 + 러너 — 4-bit NF4, 모델 1개씩 로드/언로드하며 VRAM 재사용
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_oss(model_id: str):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    kw = {'trust_remote_code': True}
    if os.environ.get('HF_TOKEN'):
        kw['token'] = os.environ['HF_TOKEN']
    tok = AutoTokenizer.from_pretrained(model_id, **kw)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, device_map='auto', quantization_config=bnb, **kw
    )
    mdl.eval()
    return tok, mdl

def run_oss(tok, mdl, prompt: str, max_new: int = 1024):
    msgs = [{'role': 'user', 'content': prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(mdl.device)
    in_len = inputs['input_ids'].shape[-1]
    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, temperature=1.0,
            pad_token_id=tok.eos_token_id,
        )
    lat = time.time() - t0
    gen = tok.decode(out[0][in_len:], skip_special_tokens=True)
    return gen, lat

def free_model(mdl, tok):
    try:
        del mdl, tok
    except Exception:
        pass
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 6) 후보 모델 리스트 — 2026-04 기준 최신 오픈소스 시리즈 (Qwen 3.5 / Gemma 4).
#    HF repo ID 가 404 로 실패하면 HF 페이지에서 공식 이름으로 교체할 것.
#    Colab T4 16GB 기준 4-bit 로 안전하게 돌아가는 후보만 기본 활성화.
OSS_MODELS = [
    'Qwen/Qwen3.5-8B-Instruct',         # 메인 후보 (Qwen 최신, ~5GB @ 4-bit)
    'Qwen/Qwen3.5-4B-Instruct',         # Qwen 경량 변종 (~2.5GB)
    'google/gemma-4-E4B-it',            # Gemma 4 경량 (~3GB)
    'google/gemma-4-E2B-it',            # symPO 현 배선된 초경량 (~2GB)
    # 'microsoft/Phi-4-mini-instruct',  # 선택: 타 패밀리 대조
    # 'meta-llama/Llama-4-Scout-17B-16E-Instruct',  # A100 전용 (MoE, VRAM 다수)
]

for mid in OSS_MODELS:
    print(f'\n=== Loading {mid} ===')
    try:
        tok, mdl = load_oss(mid)
    except Exception as e:
        print(f'  [SKIP] load failed: {type(e).__name__}: {e}')
        print(f'         → HF 페이지에서 정확한 repo ID 로 교체 후 재실행하세요.')
        continue

    rows = []
    for c in CASES:
        try:
            text, lat = run_oss(tok, mdl, c['prompt'])
        except Exception as e:
            text, lat = f'[ERROR] {type(e).__name__}: {e}', 0.0
        rows.append({'case_id': c['case_id'], 'model': mid, 'text': text, 'latency_sec': lat})
        print(f"  [{c['case_id']}] {mid.split('/')[-1]}: {lat:.2f}s, {len(text)} chars")

    ALL_RESULTS[mid] = rows
    free_model(mdl, tok)


In [ ]:
# 7) Scoring — JSON 추출 + Jaccard (vs Gemini gold)
def extract_json(text: str):
    if not text or text.startswith('[ERROR]'):
        return None
    m = re.search(r'```json\s*(.*?)```', text, re.DOTALL)
    candidate = m.group(1) if m else None
    if candidate is None:
        m2 = re.search(r'\{[\s\S]*\}', text)
        candidate = m2.group() if m2 else None
    if candidate is None:
        return None
    try:
        return json.loads(candidate)
    except Exception:
        return None

def jaccard(a, b):
    a, b = set(a or []), set(b or [])
    u = a | b
    return (len(a & b) / len(u)) if u else 1.0

def score_pair(pred, gold):
    if pred is None:
        return {'parse': 0.0, 'called_jaccard': 0.0, 'alloc_jaccard': 0.0}
    if gold is None:
        return {'parse': 1.0, 'called_jaccard': float('nan'), 'alloc_jaccard': float('nan')}
    called = jaccard(pred.get('called_agents', []), gold.get('called_agents', []))
    ap, ag = pred.get('allocations', {}) or {}, gold.get('allocations', {}) or {}
    keys = set(ap) | set(ag)
    if not keys:
        alloc = 1.0
    else:
        per = [jaccard(ap.get(k, []), ag.get(k, [])) for k in keys]
        alloc = sum(per) / len(per)
    return {'parse': 1.0, 'called_jaccard': called, 'alloc_jaccard': alloc}

# gold = gemini 응답의 파싱 JSON
gold_by_case = {r['case_id']: extract_json(r['text']) for r in ALL_RESULTS[GEMINI_MODEL]}

raw_rows = []
for model, rs in ALL_RESULTS.items():
    for r in rs:
        pred = extract_json(r['text'])
        s = score_pair(pred, gold_by_case.get(r['case_id']))
        raw_rows.append({
            'model': model, 'case_id': r['case_id'],
            'latency_sec': r['latency_sec'],
            **s,
        })

raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv('results_raw.csv', index=False)
raw_df

In [ ]:
# 8) 모델별 집계 + efficiency + 판정
agg = raw_df.groupby('model').agg(
    json_parse_rate=('parse', 'mean'),
    called_jaccard=('called_jaccard', 'mean'),
    alloc_jaccard=('alloc_jaccard', 'mean'),
    latency_sec=('latency_sec', 'mean'),
    n=('case_id', 'count'),
).reset_index()

agg['quality'] = agg[['json_parse_rate', 'called_jaccard', 'alloc_jaccard']].mean(axis=1, skipna=True)

base_q = float(agg.loc[agg['model'] == GEMINI_MODEL, 'quality'].iloc[0])
base_l = float(agg.loc[agg['model'] == GEMINI_MODEL, 'latency_sec'].iloc[0])
agg['quality_loss'] = (base_q - agg['quality']) / base_q
agg['speedup'] = base_l / agg['latency_sec']
agg['efficiency'] = agg.apply(
    lambda r: float('inf') if r['quality_loss'] <= 1e-6 else r['speedup'] / r['quality_loss'], axis=1
)

def verdict(r):
    if r['model'] == GEMINI_MODEL:
        return 'baseline'
    if r['quality_loss'] <= 0.10 and r['efficiency'] >= 2:
        return '합당'
    if r['quality_loss'] <= 0.15 and r['efficiency'] >= 1:
        return '조건부'
    return '비합당'
agg['verdict'] = agg.apply(verdict, axis=1)

agg = agg.sort_values('efficiency', ascending=False)
agg.to_csv('results_summary.csv', index=False)
agg

In [ ]:
# 9) Pareto 플롯 (Latency × Quality)
plt.figure(figsize=(9, 6))
for _, r in agg.iterrows():
    label = r['model'].split('/')[-1]
    color = 'tab:red' if r['model'] == GEMINI_MODEL else None
    plt.scatter(r['latency_sec'], r['quality'], s=160, label=label, color=color, alpha=0.8, edgecolor='black')
    plt.annotate(
        f"{label}\n({r['verdict']})",
        (r['latency_sec'], r['quality']),
        xytext=(8, 6), textcoords='offset points', fontsize=9,
    )
plt.axhline(base_q * 0.9, linestyle='--', alpha=0.4, label='품질 -10% 선')
plt.xlabel('Avg Latency (s)')
plt.ylabel('Quality (vs Gemini gold)')
plt.title(f'Supervisor task_match — Quality vs Speed  (N={len(CASES)} cases)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_quality_vs_latency.png', dpi=150)
plt.show()

print('\n=== 판정 요약 ===')
for _, r in agg.iterrows():
    print(f"{r['model']:40s}  q={r['quality']:.3f}  lat={r['latency_sec']:.2f}s  "
          f"loss={r['quality_loss']:+.1%}  speedup={r['speedup']:.2f}x  eff={r['efficiency']:.2f}  → {r['verdict']}")

## 해석 가이드

- `verdict == '합당'` 인 모델이 있으면 symPO 의 supervisor 를 해당 모델로 교체하는 것이 **데이터 기반으로 정당화**됨.
- `조건부` 는 low-stakes task (예: task_match 의 초기 분배) 에만 적용하고 `mediate` 는 frontier 유지.
- `비합당` 은 품질 손실 대비 속도 이득 부족 — 교체하지 말 것.

### 추가 실험 아이디어

1. `mediate` 케이스 추가: 여러 서브-에이전트 발언을 합의·버퍼 배정하는 프롬프트.
2. `finalize` 케이스 추가: 최종 WBS 집계.
3. Gemini 와 Claude 교차 Judge 로 rationale 품질 평가 (self-preference 편향 통제).
4. Qwen2.5-Coder-7B 추가 — 구조화 출력에 특화된 영향 분리.
5. N=10 이상으로 케이스 확장 후 bootstrap CI 보고.

### 한계

- 본 노트북은 offline 프롬프트 단위 비교. 실제 debate loop 상호작용 효과는 반영되지 않음.
- Gemini 출력을 gold 로 사용 — 절대 정답 아님. 교차 Judge 또는 structural checker 추가 권장.
- N 이 작으면 latency 분산이 큼 (Colab 공유 GPU 특성). 안정적 수치는 반복 측정 후 median 사용.